# Summary

Overall Anomaly Detection

- Goal: The AI/ML will detect the anomaly based on deviation in pattern of data from last 30 days
- How it Works: There will be score given to the anomaly based on below attributes:
    - The criticality of data
        - Minor
        - Major
        - Critical
    - Duration of anomaly
    - Frequency of anomaly in last 30 days
	
The Anomaly will be stored in MySQL/NoSQL database, with below details:
- Anomaly id
- Anomaly first occurrence
- Anomaly Duration
- Anomaly Status
- Involved network parameters
- Impacted Customers
- Impacted machines


# Preparation

## Connect Data

In [1]:
db_name= "core_stats"

In [2]:
col = "peak_upload_speed"

In [3]:
import mysql.connector
import pandas as pd
from pandas_profiling import ProfileReport

db_connection = mysql.connector.connect(
    host="20.94.254.6",
    user="gyan",
    password="5Gaa$2022",
    database="gyan_db"
)

In [4]:
mycursor = db_connection.cursor()
mycursor.execute("SELECT * FROM {} LIMIT 5".format(db_name))

myresult = mycursor.fetchall()

for x in myresult:
    print(x)

('BETAZRPDCOR001', datetime.datetime(2022, 3, 1, 14, 55, 27), 2, '0', 45751, 0, 0, 0, 0, 0, 0, 1, 2, 33.3333)
('BETAZRPDCOR001', datetime.datetime(2022, 3, 1, 15, 16, 59), 2, '0', 45901, 0, 0, 0, 0, 0, 0, 1, 2, 33.3333)
('BETAZRPDCOR001', datetime.datetime(2022, 3, 1, 15, 18, 1), 2, '0', 45901, 0, 0, 0, 0, 0, 0, 1, 2, 33.3333)
('BETAZRPDCOR001', datetime.datetime(2022, 3, 1, 15, 24, 2), 2, '0', 45921, 0, 0, 0, 0, 0, 0, 1, 2, 33.3333)
('BETAZRPDCOR001', datetime.datetime(2022, 3, 1, 15, 30, 2), 2, '0', 45940, 0, 0, 0, 0, 0, 0, 1, 2, 33.3333)


In [5]:
df = pd.read_sql('SELECT * FROM {}'.format(db_name), con=db_connection)
df.head()

,client_id,stats_timestamp,total_attached_user,total_rejected_user,peak_upload_speed,peak_download_speed,enodeb_shutdown_count,handover_failure_count,bearer_active_user_count,bearer_rejected_user_count,total_users,total_dropped_packets,enodeb_connected_count,enodeb_connection_status
0,BETAZRPDCOR001,2022-03-01 14:55:27,2,0,45751,0,0,0,0,0,0,1,2,33.3333
1,BETAZRPDCOR001,2022-03-01 15:16:59,2,0,45901,0,0,0,0,0,0,1,2,33.3333
2,BETAZRPDCOR001,2022-03-01 15:18:01,2,0,45901,0,0,0,0,0,0,1,2,33.3333
3,BETAZRPDCOR001,2022-03-01 15:24:02,2,0,45921,0,0,0,0,0,0,1,2,33.3333
4,BETAZRPDCOR001,2022-03-01 15:30:02,2,0,45940,0,0,0,0,0,0,1,2,33.3333


## Data Cleaning

In [6]:
def getSummaryTable(df, remove=False):
    '''
    Input: a Dataframe you want to get some columsn
    Output: a summary table in dataframe format, with 1st column as id, 2nd column as Missing value percentage, third column as Unique value count.

    summary_table= getSummaryTable(df)

    '''

    columns = list(df.columns)
    unique_column = [df[c].nunique() for c in columns]

    missing_number = df.isnull().sum()/len(df)
    missing_number = list(missing_number)

    Columns_Name = list(df.columns)
    summary_table = pd.DataFrame({'Columns_Name': Columns_Name,
                                 'missing_rate': missing_number,
                                  'unique_count': unique_column})
    
    if remove:
        n_row = df.shape[0]
        summary_table = summary_table[(summary_table["missing_rate"]<0.5) &(summary_table["unique_count"] !=1) & (summary_table["unique_count"] !=n_row)]

    
    return summary_table

def getDuplicateColumns(df):
    '''
    Get a list of duplicate columns.
    It will iterate over all the columns in dataframe and find the columns whose contents are duplicate.
    :param df: Dataframe object
    :return: List of columns whose contents are duplicates.
    '''
    duplicateColumnNames = set()
    # Iterate over all the columns in dataframe
    cnt = 0
    for i in range(df.shape[1]):

        # Select column at xth index.
        col = df.iloc[:, i]

        # Iterate over all the columns in DataFrame from (x+1)th index till end
        for j in range(i + 1, df.shape[1]):
            # Select column at yth index.
            otherCol = df.iloc[:, j]

            # Check if two columns at x 7 y index are equal
            if col.equals(otherCol):
                duplicateColumnNames.add(df.columns.values[j])

                print("Pair", cnt, ": ", df.columns[i], " | ", df.columns[j])
                cnt += 1

    return list(duplicateColumnNames)


def remove_duplicate_columns(df):
    '''
    Remove duplicate columns (columns with different name but same values)

    input: a dataframe
    Output: dataframe without duplicate columns

    '''

    duplicate_list = getDuplicateColumns(df)
    col = list(df.columns)
    for i in duplicate_list:
        col.remove(i)

    return df[col]

In [7]:
getSummaryTable(df)

,Columns_Name,missing_rate,unique_count
0,client_id,0.0,2
1,stats_timestamp,0.0,3497
2,total_attached_user,0.0,4
3,total_rejected_user,0.0,1
4,peak_upload_speed,0.0,1206
5,peak_download_speed,0.0,1
6,enodeb_shutdown_count,0.0,1
7,handover_failure_count,0.0,6
8,bearer_active_user_count,0.0,9
9,bearer_rejected_user_count,0.0,1


In [8]:
Summary_table= getSummaryTable(df,True)

In [9]:
getDuplicateColumns(df)

Pair 0 :  peak_download_speed  |  enodeb_shutdown_count
Pair 1 :  peak_download_speed  |  bearer_rejected_user_count
Pair 2 :  enodeb_shutdown_count  |  bearer_rejected_user_count


['enodeb_shutdown_count', 'bearer_rejected_user_count']

In [10]:
temp=df

# Starter Anomaly Detector

**How it Works?**
1. Summarize first, forecast later. 
    - Exaplin how a metric is identified as anomaly data point comparing to last 30 days.

2. Anomaly Detector Category
    - Statistical (Outlier, Z-score)
    - Time Series (Prophet, Arima | Kalman Filter)
    - Machine Learning (Clustering, Classification: Isolation Forest |)
    - Deep Learning (LSTM, Autoencoder | Clockwork RNN, Depth Gated RNN)


**How it's different from the Final Product**
1. No enough data. 
    - As for adjustments, I'll use moving window 7 days instead of 30 days for PoC.
    - e.g. What data are anomlies comparing to historical 7 (30) days
    

2. No enough column.
    - We simply want to focus on the most important column first. 
    - However, the function is designed to apply to more columns in type aspect.
    
    
3. Not Focusing on Quality, but automated process.

4. Individual labels or Scores instead of an Overall Score.

5. Keep comparison with Exiting 3rd Party Anomaly Detection tools like Anodot.


## Stats Calculations

- Mean
- Median
- IQR
- Outlier:     x> Q3+1.5IQR or x < Q1-1.5IQR 
-

**Business Explanation** 
Big jump are going on going down on upload or download speed.
We can identify there's a big jump in the 

WHen the jump happends, we send an alert.


**Column of interest:**  
- Peak download speed

### Basic Summary Stats

In [11]:
col= "peak_upload_speed"

df[col].describe()

count      3497.000000
mean      57166.980555
std       14439.126374
min          34.000000
25%       55493.000000
50%       55583.000000
75%       70891.000000
max      202333.000000
Name: peak_upload_speed, dtype: float64

### Z - Score

- Function Details
    - Input: dataframe, one column
    - Output:
        - One Score Column
            - e.g. Z-score (negative 3 to 3 )-> Adjusted Z-score ( 0~1)
        - One Label Column
        
    

In [50]:
def add_Z_score_column(df,col, z_score_threshold=2.5):
    col_name =  "Z-score_" + col
    df[col_name] = (df[col] - df[col].mean())/df[col].std(ddof=0)

    z_score_threshold = 2.5
    df["label_Z-score_"+col] = [ 1 if (x< -1*z_score_threshold or x>1*z_score_threshold) else 0 for x in df[col_name]]
    return df

In [32]:
add_Z_score_column(df,col)

### Outlier

In [49]:
from scipy.stats import iqr
import numpy as np

#outlier   q3+1.5IQR   Q1-1.5IQR


def add_outlier_column(df, col, iqr_factor=1.5, low_quantile=25, upper_quantile=75):
    q3, q1 = np.percentile(df[col], [upper_quantile, low_quantile])

    iqr_v = iqr(df[col])

    upper_v = q3+iqr_factor*iqr_v
    lower_v = q1-iqr_factor*iqr_v
    
    #outliers_removed = [x for x in df[col] if x >= lower_v and x <= upper_v]
    #print(outliers_removed)
    
    outlier_index = df[(df[col] > upper_v) | (df[col] < lower_v)][col].index
    outlier_label = [1 if x in outlier_index else 0 for x in df.index]
    
    df["label_outlier_"+col]  = outlier_label
    return df

In [34]:
add_outlier_column(df,col)

### Apply Stats Functions for All column in one Database

In [44]:
keeped_column_name = list(Summary_table["Columns_Name"])
keeped_column_name

['client_id',
 'total_attached_user',
 'peak_upload_speed',
 'handover_failure_count',
 'bearer_active_user_count',
 'total_users',
 'total_dropped_packets',
 'enodeb_connected_count',
 'enodeb_connection_status']

In [54]:
keeped_column_name.remove("client_id")


ValueError: list.remove(x): x not in list

In [47]:
Stats_summary_raw = df[keeped_column_name]

In [53]:
for col in keeped_column_name:
    
    Stats_summary_raw=add_Z_score_column(Stats_summary_raw,col)
    Stats_summary_raw=add_outlier_column(Stats_summary_raw,col)

## Questions

1. In the Last 30 days, for peak_upload_speed.

    - What is normal data? 
    - How many anomaly data point appears
    - The criticality of data
        - Minor
        - Major
        - Critical
    - Duration of anomaly
    - Frequency of anomaly in last 30 days

### What is normal data? 

In [16]:
col= "peak_upload_speed"
n_row =df.shape[0]
df[col].describe()

count      3497.000000
mean      57166.980555
std       14439.126374
min          34.000000
25%       55493.000000
50%       55583.000000
75%       70891.000000
max      202333.000000
Name: peak_upload_speed, dtype: float64

In [17]:
df.columns

Index(['client_id', 'stats_timestamp', 'total_attached_user',
       'total_rejected_user', 'peak_upload_speed', 'peak_download_speed',
       'enodeb_shutdown_count', 'handover_failure_count',
       'bearer_active_user_count', 'bearer_rejected_user_count', 'total_users',
       'total_dropped_packets', 'enodeb_connected_count',
       'enodeb_connection_status', 'Z-score_peak_upload_speed',
       'label_Z-score_peak_upload_speed', 'label_outlierpeak_upload_speed'],
      dtype='object')

In [20]:
Z_score_num_anomaly= round(df["label_Z-score_"+col].sum()/n_row*100,1)
print("Q: How many data are classified as anomaly based on Z-score rule","\nA:",Z_score_num_anomaly,"%")

Q: How many data are classified as anomaly based on Z-score rule 
A: 4.3 %


In [23]:
outlier_num_anomaly= round(df["label_outlier_"+col].sum()/n_row*100,1)
print("Q: How many data are classified as anomaly based on Outlier rule","\nA:",outlier_num_anomaly,"%")

Q: How many data are classified as anomaly based on Outlier rule 
A: 6.3 %


In [28]:
outlier_num_anomaly= round(df["label_outlier_"+col].sum()/n_row*100,1)
print("Q: What's the Duration of Anomaly","\nA:",outlier_num_anomaly,"%")

Q: What's the Duration of Anomaly 
A: 6.3 %


### Datetime

In [29]:
from datetime import date

today = date.today()

In [30]:
today

datetime.date(2022, 3, 14)

In [ ]:
df["stats_timestamp"][0]

In [ ]:
label_columns = df.columns[list(map(lambda x: x.startswith('label'),df))]

### Data Visualization for the PoC

In [ ]:
# importing minmax scaler 
from sklearn.preprocessing import MinMaxScaler
# model creation 
min_max_scaler = MinMaxScaler()
# fitting and transforming the model on A(train_inputs)


#### Time Value based Plot

In [ ]:
import matplotlib

import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
df["timestamp"]=df.stats_timestamp
plt.rcParams["figure.figsize"] = (20,20)

# change the type of timestamp column for plotting
df['timestamp'] = pd.to_datetime(df['timestamp'])

# plot the data
df.plot(x='timestamp', y=col)

### Distribution Plot

In [ ]:
import plotly.figure_factory as ff
import numpy as np

x1 = np.random.randn(200)
x2 = np.random.randn(200) + 2

group_labels = ['Group 1', 'Group 2']

colors = ['slategray', 'magenta']

# Create distplot with curve_type set to 'normal'
fig = ff.create_distplot([x1, x2], group_labels, bin_size=.5,
                         curve_type='normal' # override default 'kde'
                         )

# Add title
fig.update_layout(title_text='Distplot with Normal Distribution')
fig.show()

### Box Plot

In [ ]:
# Default Box plot do not work well, we need data transformation

plt.rcParams["figure.figsize"] = (5,20)



sns.boxplot(data=df[col])

In [ ]:
min_max_diff = df[col].max() - df[col].min()

In [ ]:
[x-df[col].min() for x in df[col]]

In [ ]:
# # change the type of timestamp column for plotting
# df['timestamp'] = pd.to_datetime(df['timestamp'])
# # change fahrenheit to °C (temperature mean= 71 -> fahrenheit)
# df['value'] = (df['value'] - 32) * 5/9
# # plot the data
# df.plot(x='timestamp', y='value')

### Normality Assumption

In [ ]:
!pip install plotly

In [ ]:
upload_freq_table = pd.Series(df[col]).value_counts().reset_index().sort_values('index').reset_index(drop=True)
upload_freq_table.columns = ['upload_value', 'Frequency']

In [ ]:
upload_freq_table.upload_value

In [ ]:
import plotly.figure_factory as ff
import numpy as np
np.random.seed(1)


hist_data = [upload_freq_table.upload_value, upload_freq_table.Frequency]


In [ ]:
import collections
freq_tb = collections.Counter(df[col])

In [ ]:
# group_labels = ['distplot'] # name of the dataset

# fig = ff.create_distplot(hist_data, group_labels)
# fig.show()

In [ ]:
# import plotly.express as px
# df = px.data.tips()
# fig = px.histogram(df, x=col, y="tip", color="sex", marginal="rug",
#                    hover_data=df.columns)
# fig.show()

## Feature Engineering

In [ ]:
# the hours and if it's night or day (7:00-22:00)
df['hours'] = df['timestamp'].dt.hour
df['daylight'] = ((df['hours'] >= 7) & (df['hours'] <= 22)).astype(int)

In [ ]:
# the day of the week (Monday=0, Sunday=6) and if it's a week end day or week day.
df['DayOfTheWeek'] = df['timestamp'].dt.dayofweek
df['WeekDay'] = (df['DayOfTheWeek'] < 5).astype(int)
# An estimation of anomly population of the dataset (necessary for several algorithm)
outliers_fraction = 0.01

In [ ]:
# time with int to plot easily
df['time_epoch'] = (df['timestamp'].astype(np.int64)/100000000000).astype(np.int64)

In [ ]:
# # creation of 4 distinct categories that seem useful (week end/day week & night/day)
# df['categories'] = df['WeekDay']*2 + df['daylight']

# a = df.loc[df['categories'] == 0, 'value']
# b = df.loc[df['categories'] == 1, 'value']
# c = df.loc[df['categories'] == 2, 'value']
# d = df.loc[df['categories'] == 3, 'value']

# fig, ax = plt.subplots()
# a_heights, a_bins = np.histogram(a)
# b_heights, b_bins = np.histogram(b, bins=a_bins)
# c_heights, c_bins = np.histogram(c, bins=a_bins)
# d_heights, d_bins = np.histogram(d, bins=a_bins)

# width = (a_bins[1] - a_bins[0])/6

# ax.bar(a_bins[:-1], a_heights*100/a.count(), width=width, facecolor='blue', label='WeekEndNight')
# ax.bar(b_bins[:-1]+width, (b_heights*100/b.count()), width=width, facecolor='green', label ='WeekEndLight')
# ax.bar(c_bins[:-1]+width*2, (c_heights*100/c.count()), width=width, facecolor='red', label ='WeekDayNight')
# ax.bar(d_bins[:-1]+width*3, (d_heights*100/d.count()), width=width, facecolor='black', label ='WeekDayLight')

# plt.legend()
# plt.show()

## Time Series

## Functions need  exploring

In [ ]:
#https://github.com/Vicam/Unsupervised_Anomaly_Detection/blob/master/custom_function.py
#from pyemma import msm
import pandas as pd
import numpy as np

# return Series of distance between each point and his distance with the closest centroid
def getDistanceByPoint(data, model):
    distance = pd.Series()
    for i in range(0,len(data)):
        Xa = np.array(data.loc[i])
        Xb = model.cluster_centers_[model.labels_[i]-1]
        distance.set_value(i, np.linalg.norm(Xa-Xb))
    return distance

# train markov model to get transition matrix
# def getTransitionMatrix (df):
#     df = np.array(df)
#     model = msm.estimate_markov_model(df, 1)
#     return model.transition_matrix

# return the success probability of the state change 
def successProbabilityMetric(state1, state2, transition_matrix):
    proba = 0
    for k in range(0,len(transition_matrix)):
        if (k != (state2-1)):
            proba += transition_matrix[state1-1][k]
    return 1-proba

# return the success probability of the whole sequence
def sucessScore(sequence, transition_matrix):
    proba = 0 
    for i in range(1,len(sequence)):
        if(i == 1):
            proba = successProbabilityMetric(sequence[i-1], sequence[i], transition_matrix)
        else:
            proba = proba*successProbabilityMetric(sequence[i-1], sequence[i], transition_matrix)
    return proba

# return if the sequence is an anomaly considering a threshold
def anomalyElement(sequence, threshold, transition_matrix):
    if (sucessScore(sequence, transition_matrix) > threshold):
        return 0
    else:
        return 1

# return a dataframe containing anomaly result for the whole dataset 
# choosing a sliding windows size (size of sequence to evaluate) and a threshold
def markovAnomaly(df, windows_size, threshold):
    transition_matrix = getTransitionMatrix(df)
    real_threshold = threshold**windows_size
    df_anomaly = []
    for j in range(0, len(df)):
        if (j < windows_size):
            df_anomaly.append(0)
        else:
            sequence = df[j-windows_size:j]
            sequence = sequence.reset_index(drop=True)
            df_anomaly.append(anomalyElement(sequence, real_threshold, transition_matrix))
    return df_anomaly

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


# multiply and add by random numbers to get some real values
data = np.random.randn(50000)  * 20 + 20

# Function to Detection Outlier on one-dimentional datasets.
def find_anomalies(data):
    #define a list to accumlate anomalies
    anomalies = []
    
    # Set upper and lower limit to 3 standard deviation
    data_std = np.std(data)
    data_mean = np.mean(data)
    anomaly_cut_off = data_std * 3
    
    lower_limit  = data_mean - anomaly_cut_off 
    upper_limit = data_mean + anomaly_cut_off
    print(lower_limit)
    # Generate outliers
    
    for outlier in data:
        if outlier > upper_limit or outlier < lower_limit:
            anomalies.append(outlier)
    return anomalies

find_anomalies(data)